# 04.4 GPU 与训练效率（GPU and Efficiency）

这一节的目标不是做极限性能优化，而是先建立几条最有用的效率直觉。  

重点概念

- 设备`
- 批大小`
- 固定内存`
- 非阻塞拷贝`
- 自动混合精度`

## 学习目标

学完后你应该能

1. 正确地把模型和数据放到同一个 `device` 上
2. 理解 `batch size` 对 step 数量和训练耗时的影响
3. 理解为什么推理时常用 `model.eval()` 和 `torch.no_grad()`
4. non_blocking` 在 GPU 训练中的典型用途
5. 对 `automatic mixed precision
6. 形成一个足够实用的性能检查清单

In [ ]:
import time

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)
print("cuda available / 是否可用 CUDA =", torch.cuda.is_available())

## 设备基础

最基本也最容易出错的一点是：  

- model 和 data 必须在同一个设备上

In [ ]:
class TinyNet(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = TinyNet().to(device)
x_cpu = torch.randn(4, 128)
y_cpu = torch.tensor([0, 1, 0, 1], dtype=torch.long)

x = x_cpu.to(device)
y = y_cpu.to(device)
logits = model(x)

print("model device =", next(model.parameters()).device)
print("x device =", x.device)
print("y device =", y.device)
print("logits.shape =", logits.shape)

如果你看到 `Expected all tensors to be on the same device` 这种错误，优先排查这里。  


## 批大小 的影响

`batch size` 会直接影响每个 epoch 的 step 数量。  

通常来说

- batch 更大，step 更少
- 内存占用更高
- 不是越大越好

In [ ]:
features = torch.randn(4096, 128)
labels = (features[:, 0] + 0.3 * features[:, 1] > 0).long()
dataset = TensorDataset(features, labels)


def sync_if_needed():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def benchmark_one_epoch(batch_size, device):
    torch.manual_seed(42)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == "cuda"),
    )
    model = TinyNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()

    sync_if_needed()
    start = time.perf_counter()
    total_items = 0
    total_loss = 0.0
    num_steps = 0

    model.train()
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=(device.type == "cuda"))
        yb = yb.to(device, non_blocking=(device.type == "cuda"))
        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_items += xb.size(0)
        total_loss += loss.item() * xb.size(0)
        num_steps += 1

    sync_if_needed()
    elapsed = time.perf_counter() - start
    return {
        "batch_size": batch_size,
        "steps_per_epoch": num_steps,
        "elapsed_sec": round(elapsed, 4),
        "items_per_sec": round(total_items / elapsed, 2),
        "avg_loss": round(total_loss / total_items, 4),
    }


batch_results = [benchmark_one_epoch(bs, device) for bs in [16, 64, 256]]
batch_df = pd.DataFrame(batch_results)
batch_df

你重点要看两列

- `steps_per_epoch`
- `elapsed_sec`

真实项目里还要同时考虑：显存限制、收敛稳定性、吞吐量。  


## 推理效率

推理时常见的两个动作是：  

- `model.eval()`
- `torch.no_grad()`

BatchNorm` 这类层的行为；`no_grad()` 会避免构建梯度图。  


In [ ]:
inference_model = TinyNet(hidden_dim=256).to(device)
inference_model.eval()
input_batch = torch.randn(2048, 128).to(device)
original_num_threads = torch.get_num_threads()
torch.set_num_threads(1)


def benchmark_inference(model, x, use_no_grad, repeats=120, warmup=20):
    context = torch.no_grad() if use_no_grad else torch.enable_grad()

    with context:
        for _ in range(warmup):
            out = model(x)

    sync_if_needed()
    start = time.perf_counter()
    with context:
        for _ in range(repeats):
            out = model(x)
    sync_if_needed()
    return time.perf_counter() - start, out.requires_grad


time_with_grad, grad_flag = benchmark_inference(inference_model, input_batch, use_no_grad=False)
time_no_grad, no_grad_flag = benchmark_inference(inference_model, input_batch, use_no_grad=True)
torch.set_num_threads(original_num_threads)

print("output.requires_grad with grad mode =", grad_flag)
print("output.requires_grad with no_grad =", no_grad_flag)
print("time_with_grad =", round(time_with_grad, 4), "sec")
print("time_no_grad =", round(time_no_grad, 4), "sec")
print("speedup ratio / 提速比 =", round(time_with_grad / max(time_no_grad, 1e-8), 3))

`no_grad()` 不是“永远显著更快”的魔法开关，但它会明确关闭梯度图构建。  

在更大的模型、GPU 推理或更长的批处理上，它通常更省内存，也往往更高效。  


## 数据加载设置

你不需要一开始就把 `num_workers` 调得很复杂，但应该知道这些参数的意图。  


In [ ]:
def suggested_loader_kwargs(device):
    if device.type == "cuda":
        return {
            "num_workers": 2,
            "pin_memory": True,
            "note": "GPU training usually benefits from pinned memory and background workers.",
        }
    return {
        "num_workers": 0,
        "pin_memory": False,
        "note": "CPU or debugging mode usually starts with a simpler loader configuration.",
    }


loader_kwargs = suggested_loader_kwargs(device)
loader_kwargs

一个够实用的记忆方式

- `pin_memory=True`：更适合 CPU -> GPU 数据拷贝
- `non_blocking=True`：常和 pinned memory 搭配
- `num_workers`：让数据准备和训练更并行

## 自动混合精度

自动混合精度` 的直觉是：  

- 让一部分计算用更低精度完成
- 从而换取更好的吞吐量和更低显存占用

这通常主要在 CUDA 训练里发挥作用。  


In [ ]:
amp_loader = DataLoader(dataset, batch_size=128, shuffle=True, pin_memory=(device.type == "cuda"))
amp_model = TinyNet().to(device)
amp_loss_fn = nn.CrossEntropyLoss()
amp_optimizer = torch.optim.Adam(amp_model.parameters(), lr=0.01)

if device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    xb, yb = next(iter(amp_loader))
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)

    amp_model.train()
    amp_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = amp_model(xb)
        loss = amp_loss_fn(logits, yb)

    scaler.scale(loss).backward()
    scaler.step(amp_optimizer)
    scaler.update()

    print("AMP step finished / AMP 训练步已完成")
    print("loss =", float(loss))
else:
    print("CUDA not available / 当前环境没有可用 CUDA，所以跳过真实 AMP 演示。")
    print("但你应该记住：AMP 常见于 GPU 训练，用来提高吞吐量并降低显存占用。")

In [ ]:
# 练习 1
# 为什么 batch size 变大时，steps_per_epoch 往往会变少？
# Why does steps_per_epoch usually become smaller when batch size becomes larger?

练习 1 参考答案

因为每一步处理的样本更多了，所以同样数量的数据需要更少的 step。  

但这不代表 batch 一定越大越好，因为还要考虑显存和优化行为。  


In [ ]:
# 练习 2
# 推理时为什么常常要同时写 model.eval() 和 torch.no_grad()？
# Why do we often use both model.eval() and torch.no_grad() during inference?

练习 2 参考答案

- `model.eval()`：让 `Dropout（BatchNorm` into inference mode）
- `torch.no_grad()`：避免构建梯度图，减少额外开销

这两者解决的问题不同，所以经常一起用。  


## 6. 小结

这一节最重要的收获

1. model 和 data 必须放在同一个 `device` 上
2. `batch size` 会影响 step 数量、吞吐量和内存占用
3. 推理时通常使用 `model.eval()` + `torch.no_grad()`
4. non_blocking` 更偏向 GPU 数据传输优化
5. 自动混合精度` 是常见的 GPU 训练提速手段之一